# Desproporcionalidade

In [1]:
import sys
import pandas as pd
sys.path.append('../../src/')

# Configurações de exibição
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 100
%config SqlMagic.named_parameters = "enabled"
%config SqlMagic.autocommit = True

# Banco em arquivo (persiste durante a sessão).
#%sql duckdb:///content/vigimed.duckdb
# Se preferir memória: 
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

# 1. Iniciando a seção

In [2]:
%sql ROLLBACK;

Running query in 'duckdb:///:memory:'

,Success


# 2. Criando as tabelas

In [12]:
%%sql
DROP TABLE IF EXISTS dim_atc;
CREATE TABLE dim_atc AS
SELECT * FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\dim_atc\dim_atc.parquet');


DROP TABLE IF EXISTS fat_medicamentos;
CREATE TABLE fat_medicamentos AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\fat_medicamentos\fat_medicamentos.parquet');

DROP TABLE IF EXISTS dim_notificacoes;
CREATE TABLE dim_notificacoes AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\dim_notificacoes\dim_notificacoes.parquet');

DROP TABLE IF EXISTS fat_reacoes;
CREATE TABLE fat_reacoes AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\fat_reacoes\fat_reacoes.parquet');
 

Running query in 'duckdb:///:memory:'

,Success


# 5. Classificando atc_level_5 para estudo

In [17]:
%%sql
DROP TABLE IF EXISTS fat_med_w_atc_level_5;

CREATE TABLE fat_med_w_atc_level_5 AS
with fat_med as (
SELECT 
    fat_medicamentos.*, atc.ATC_CODE_LEVEL_4_LEVEL_NAME,
    (
        CASE 
            WHEN atc.ATC_CODE_LEVEL_4 IN ('L04AB', 'L01FA', 'L04AC','L04AF') 
                THEN (
                    CASE 
                        /*Rituximab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%rituximab%' THEN 'L01FA01_Rituximab'
                        /*Abatacept*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%abatacept%' THEN 'L04AA24_Abatacept'
                        /*Etanercept*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%etanercept%' THEN 'L04AB01_Etanercept'
                        /*Infliximab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%infliximab%' THEN 'L04AB02_Infliximab'
                        /*Adalimumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%adalimumab%' THEN 'L04AB04_Adalimumab'
                        /*Certolizumab Pegol*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab%' THEN 'L04AB05_Certolizumab Pegol'   
                        -- WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab pegol%' THEN 'L04AB05_Certolizumab Pegol'
                        -- WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumabe pegol%' THEN 'L04AB05_Certolizumab Pegol'
                        
                        /*Golimumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%golimumab%' THEN 'L04AB06_Golimumab'
                        /*Tocilizumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%tocilizumab%' THEN 'L04AC07_Tocilizumab'
                        /*Baricitinib */
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%baricitinib%' THEN 'L04AF02_Baricitinib'
                        /*Upadacitinib*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%upadacitinib%' THEN 'L04AF03_Upadacitinib' 
                        ELSE 'missing'
                    END
                )
            ELSE 'Outros'
        END
    ) AS PRINCIPIOS_ATIVOS_CLEAN
FROM fat_medicamentos
INNER JOIN dim_atc AS atc
    ON fat_medicamentos.ATC_CODE_LEVEL_4 = atc.ATC_CODE_LEVEL_4 
    AND atc.ATC_LEVEL = 4
)
select 
    (case when PRINCIPIOS_ATIVOS_CLEAN in ('missing','Outros') then 'Outros' 
        else concat(ATC_CODE_LEVEL_4,'_',ATC_CODE_LEVEL_4_LEVEL_NAME) end) as ATC_LEVEL_4_NAME
,*
from fat_med

Running query in 'duckdb:///:memory:'

,Success


## Analytic Query

In [ ]:
%%sql

with dim_id_pesquisa as (
    select distinct IDENTIFICACAO_NOTIFICACAO 
    from fat_med_w_atc_level_5
    where ATC_LEVEL_4_NAME not in ('Outros')
)

select * 
from dim_notificacoes_estudo
LIMIT 5



In [ ]:
%sql
-- Tabelas
-- fat_med_not
-- fat_reacoes

-- Colunas (substitua nos SELECTs abaixo):
-- IDENTIFICACAO_NOTIFICACAO  -> id
-- nome_medicamento           -> medicamento
-- medicamento_estudo         -> med_flag
-- PT (ou HLT/HLGT/SOC/LLT)   -> evento
-- SOC                        -> SOC
-- current_flag               -> rx_current

-- Parâmetros “manuais”:
-- only_medicamento_estudo: TRUE/FALSE
-- use_target_list: TRUE/FALSE
-- target_medicamentos: ('L01FA01_rituximab','metformina',...)
-- use_current_flag: TRUE/FALSE
-- min_cases: 3

WITH
populacao AS (
  SELECT *
  FROM dim_notificacoes
  WHERE sexo='M' AND idade BETWEEN 18 AND 65
),
meds AS (
  SELECT *
  FROM fat_med_not
  inner join populacao using (IDENTIFICACAO_NOTIFICACAO)
  WHERE 
),
rx_base AS (
  SELECT *
  FROM fat_reacoes
  inner join populacao using (id)
  WHERE evento IS NOT NULL
    AND (TRUE = FALSE /*use_current_flag*/ OR rx_current = TRUE)
),
rx_pairs AS (
  SELECT DISTINCT
    id,
    evento
  FROM rx_base
),
event_soc AS (
  SELECT evento, MIN(SOC) AS SOC
  FROM rx_base
  WHERE SOC IS NOT NULL
  GROUP BY evento
),
universe AS (
  SELECT COUNT(DISTINCT id) AS total_ids
  FROM rx_pairs
),
event_ids AS (
  SELECT evento, COUNT(DISTINCT id) AS total_ev_ids
  FROM rx_pairs
  GROUP BY evento
),
exposed AS (
  SELECT DISTINCT medicamento, id
  FROM meds
),
exposed_in_universe AS (
  SELECT e.medicamento, e.id
  FROM exposed e
  JOIN (SELECT DISTINCT id FROM rx_pairs) u USING (id)
),
N1_by_med AS (
  SELECT medicamento, COUNT(DISTINCT id) AS N1
  FROM exposed_in_universe
  GROUP BY medicamento
),
a_counts AS (
  SELECT
    eiu.medicamento,
    r.evento,
    COUNT(DISTINCT r.id) AS a
  FROM exposed_in_universe eiu
  JOIN rx_pairs r USING (id)
  GROUP BY eiu.medicamento, r.evento
),
med_event AS (
  SELECT DISTINCT n.medicamento, e.evento
  FROM N1_by_med n
  CROSS JOIN event_ids e
),
calc AS (
  SELECT
    me.medicamento,
    es.SOC,
    me.evento,
    COALESCE(a.a, 0) AS a,
    n.N1 AS N1,
    (u.total_ids - n.N1) AS N0,
    (e.total_ev_ids - COALESCE(a.a,0)) AS c,
    (n.N1 - COALESCE(a.a,0)) AS b,
    ((u.total_ids - n.N1) - (e.total_ev_ids - COALESCE(a.a,0))) AS d
  FROM med_event me
  JOIN N1_by_med n USING (medicamento)
  JOIN event_ids e USING (evento)
  CROSS JOIN universe u
  LEFT JOIN a_counts a
    ON a.medicamento = me.medicamento AND a.evento = me.evento
  LEFT JOIN event_soc es
    ON es.evento = me.evento
  WHERE n.N1 > 0 AND (u.total_ids - n.N1) > 0
),
final AS (
  SELECT
    medicamento,
    SOC,
    evento,
    a, b, c, d,
    N1 AS N_medicamento,
    N0 AS N_outros,
    CASE WHEN a=0 OR b=0 OR c=0 OR d=0 THEN a+0.5 ELSE a END AS A_,
    CASE WHEN a=0 OR b=0 OR c=0 OR d=0 THEN b+0.5 ELSE b END AS B_,
    CASE WHEN a=0 OR b=0 OR c=0 OR d=0 THEN c+0.5 ELSE c END AS C_,
    CASE WHEN a=0 OR b=0 OR c=0 OR d=0 THEN d+0.5 ELSE d END AS D_
  FROM calc
)
SELECT
  medicamento,
  SOC,
  evento,
  a, b, c, d,
  N_medicamento,
  N_outros,
  (A_ * D_) / NULLIF(B_ * C_, 0)                                AS ROR,
  EXP(LN((A_ * D_) / NULLIF(B_ * C_, 0)) - 1.96*SQRT(1/A_+1/B_+1/C_+1/D_)) AS IC95_low,
  EXP(LN((A_ * D_) / NULLIF(B_ * C_, 0)) + 1.96*SQRT(1/A_+1/B_+1/C_+1/D_)) AS IC95_high,
  CASE
    WHEN a >= 3 /*min_cases*/
     AND EXP(LN((A_ * D_) / NULLIF(B_ * C_, 0)) - 1.96*SQRT(1/A_+1/B_+1/C_+1/D_)) > 1.0
    THEN TRUE ELSE FALSE
  END AS sinal
FROM final
ORDER BY medicamento ASC, sinal DESC, ROR DESC, a DESC;